# Notebook 3 — Transpiled QAOA (Aer + FakeSherbrooke)

Validates the noise model: F_p^noisy ≈ (1−ε)^G_p · F_p + [1−(1−ε)^G_p] · C̄

Locates the empirical optimal depth p* and compares to the theoretical prediction.

In [ ]:
import numpy as np, networkx as nx, matplotlib.pyplot as plt, time, warnings
warnings.filterwarnings('ignore')
from scipy.optimize import minimize
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

try:
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
    BACKEND     = FakeSherbrooke()
    NOISE_MODEL = NoiseModel.from_backend(BACKEND)
    PM          = generate_preset_pass_manager(optimization_level=3, backend=BACKEND)
    USE_FAKE    = True
    print(f"FakeSherbrooke: {BACKEND.num_qubits} qubits")
except Exception as e:
    print(f"FakeSherbrooke unavailable ({e}); using manual depolarising noise")
    BACKEND=None; NOISE_MODEL=None; PM=None; USE_FAKE=False

from qaoa import (
    build_cost_hamiltonian, build_qaoa_circuit_parallel,
    brute_force_maxcut, uniform_cut_expectation, circuit_stats,
    optimise_qaoa, interp_init, AdaptiveShotSchedule,
)

SEED=42; np.random.seed(SEED)
EPS = 2e-3   # per-gate depolarising rate (FakeSherbrooke calibration)

def build_noise_model_manual(eps_1q=1e-4, eps_2q=EPS):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(eps_1q,1), ['rz','rx','h','x','sx'])
    nm.add_all_qubit_quantum_error(depolarizing_error(eps_2q,2), ['cx','ecr'])
    return nm

print(f"Depolarising ε = {EPS}")

In [ ]:
from qiskit import transpile

G_test  = nx.random_regular_graph(3, 6, seed=SEED)
H_test  = build_cost_hamiltonian(G_test)
opt_val, _ = brute_force_maxcut(G_test)
C_bar   = uniform_cut_expectation(G_test)
nodes   = list(G_test.nodes())
noise_model = NOISE_MODEL if USE_FAKE else build_noise_model_manual()
sim = AerSimulator(noise_model=noise_model)

print(f"Graph n={G_test.number_of_nodes()}, MaxCut={opt_val}, C̄={C_bar:.1f}")

Fp_ideal, Fp_noisy, Fp_pred, G_ps = [], [], [], []
gamma_prev, beta_prev = None, None

for p in [1,2,3,4,5]:
    qc_sym = build_qaoa_circuit_parallel(G_test, p)

    # Transpile
    if USE_FAKE:
        qc_t = PM.run(qc_sym)
    else:
        qc_t = transpile(qc_sym, basis_gates=['cx','rz','rx','h','x'],
                         optimization_level=3)
    G_p = qc_t.count_ops().get('cx',0)+qc_t.count_ops().get('ecr',0)
    G_ps.append(G_p)

    # INTERP init
    if gamma_prev is not None:
        g0,b0 = interp_init(gamma_prev, beta_prev)
        init  = np.concatenate([g0,b0])
    else:
        init = np.random.uniform(0,np.pi,2*p)

    # Ideal optimum
    def ideal_obj(params):
        return -Statevector(build_qaoa_circuit_parallel(G_test,p,bind_params=params)).expectation_value(H_test).real
    res_ideal = minimize(ideal_obj, init, method='COBYLA', options={'maxiter':400,'rhobeg':0.5})
    Fp_i      = -res_ideal.fun
    gamma_prev= res_ideal.x[:p]; beta_prev=res_ideal.x[p:]
    Fp_ideal.append(Fp_i)

    # Noisy evaluation at ideal optimum
    qc_bound = qc_t.assign_parameters(
        dict(zip(sorted(qc_t.parameters,key=lambda x:x.name), list(res_ideal.x))))
    qc_m = qc_bound.copy(); qc_m.measure_all()
    counts = sim.run(qc_m, shots=2048).result().get_counts()
    total  = sum(counts.values()); Fp_n = 0.0
    for bitstr, cnt in counts.items():
        z = {nodes[i]: int(b)
             for i,b in enumerate(reversed(bitstr[:G_test.number_of_nodes()]))}
        cv = sum(G_test[u][v].get('weight',1.0) for u,v in G_test.edges()
                 if z.get(u,0)!=z.get(v,0))
        Fp_n += cnt/total*cv
    Fp_noisy.append(Fp_n)

    # Model prediction (Eq. 7.1 of paper)
    Fp_pr = (1-EPS)**G_p * Fp_i + (1-(1-EPS)**G_p) * C_bar
    Fp_pred.append(Fp_pr)

    print(f"p={p}: G_p={G_p:3d}, F_ideal={Fp_i:.4f}, F_noisy={Fp_n:.4f}, "
          f"F_pred={Fp_pr:.4f}, α={Fp_i/opt_val:.3f}")

p_star = [1,2,3,4,5][np.argmax(Fp_noisy)]
print(f"\nEmpirical optimal depth p* = {p_star}")
print(f"Theoretical p* ≈ 1/(2·ḡ·ε) where ḡ = {np.mean(G_ps)/(np.mean([1,2,3,4,5])):.1f}")

In [ ]:
depths = [1,2,3,4,5]
fig, axes = plt.subplots(1,2,figsize=(11,4))

axes[0].plot(depths, [v/opt_val for v in Fp_ideal], 'ro-', lw=2, ms=7, label='Ideal')
axes[0].plot(depths, [v/opt_val for v in Fp_noisy], 'bs-', lw=2, ms=7, label='Noisy (Aer)')
axes[0].plot(depths, [v/opt_val for v in Fp_pred],  'g^--',lw=1.5,ms=6, label='Model prediction')
axes[0].axvline(p_star, ls=':', color='orange', lw=2, label=f'p*={p_star}')
axes[0].set_xlabel('Depth p'); axes[0].set_ylabel('Approximation ratio')
axes[0].set_title('Noise-depth tradeoff'); axes[0].legend(fontsize=9)

axes[1].bar(depths, G_ps, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Depth p'); axes[1].set_ylabel('Two-qubit gate count G_p')
axes[1].set_title('Post-transpilation ECR/CX count')
plt.tight_layout(); plt.show()

# Validate noise model
print("\nNoise model validation (F_noisy vs prediction):")
mae = np.mean(np.abs(np.array(Fp_noisy)-np.array(Fp_pred)))
print(f"MAE = {mae:.4f}  (small → model holds)")